In [1]:
from pathlib import Path
import sys

sys.path.append(str(Path.cwd().parent))

In [2]:
from src.transforms import trainval_transforms, revert_normalization, revert_standardization
from src.dataset import ImageDataset
import torch

annot_path = Path("../data/preprocessed/trainval/annotations.csv")
img_dir = Path("../data/preprocessed/trainval/images")

# test dataset without transforms
dataset = ImageDataset(annot_path, img_dir, transform=trainval_transforms)

In [3]:
from src.model import Model
from torch.utils.data import DataLoader

trainval_dl = DataLoader(dataset, 8, True)
X_batch, y_batch = next(iter(trainval_dl))
model = Model()

model.eval()
with torch.no_grad():
    preds_batch = model(X_batch)

In [4]:
from src.postprocessing import postprocess_preds

postprocessed_preds_1 = postprocess_preds(preds_batch[0])
postprocessed_preds_2 = postprocess_preds(preds_batch[1])

In [5]:
postprocessed_preds_1

{'tvmonitor': [(tensor(0.0003),
   tensor(-0.8924),
   tensor(-1.4371),
   tensor(1.9218),
   tensor(0.4367)),
  (tensor(6.9343e-05),
   tensor(-1.5245),
   tensor(34.6916),
   tensor(1.6347),
   tensor(28.3108)),
  (tensor(2.5907e-05),
   tensor(159.7976),
   tensor(65.5800),
   tensor(160.9863),
   tensor(62.5479))],
 'pottedplant': [(tensor(0.0005),
   tensor(1.3062),
   tensor(96.2089),
   tensor(-2.2315),
   tensor(96.4376)),
  (tensor(0.0003),
   tensor(95.7458),
   tensor(29.7548),
   tensor(97.4945),
   tensor(33.2945)),
  (tensor(0.0001),
   tensor(66.3264),
   tensor(0.3582),
   tensor(62.0167),
   tensor(0.0039))],
 'train': [(tensor(0.0002),
   tensor(94.7316),
   tensor(0.5789),
   tensor(96.4186),
   tensor(0.3620))],
 'sheep': [(tensor(1.6834e-05),
   tensor(126.4282),
   tensor(1.2802),
   tensor(129.2393),
   tensor(-0.1858))],
 'cow': [(tensor(0.0003),
   tensor(157.8449),
   tensor(97.4310),
   tensor(161.3764),
   tensor(95.5117)),
  (tensor(4.3131e-05),
   tensor(1

In [6]:
postprocessed_preds_2

{'tvmonitor': [(tensor(0.0003),
   tensor(-0.8784),
   tensor(-1.3549),
   tensor(1.9020),
   tensor(0.2551)),
  (tensor(7.2800e-05),
   tensor(-1.4303),
   tensor(34.4880),
   tensor(1.5810),
   tensor(28.3871))],
 'pottedplant': [(tensor(0.0005),
   tensor(1.4782),
   tensor(96.3350),
   tensor(-2.2833),
   tensor(96.2223)),
  (tensor(0.0004),
   tensor(95.7482),
   tensor(29.7030),
   tensor(97.5807),
   tensor(33.3953)),
  (tensor(0.0001),
   tensor(66.4062),
   tensor(0.2834),
   tensor(61.8330),
   tensor(0.0737))],
 'train': [(tensor(0.0002),
   tensor(94.5845),
   tensor(0.4487),
   tensor(96.6162),
   tensor(0.4644))],
 'sheep': [(tensor(2.9600e-05),
   tensor(126.1729),
   tensor(1.4568),
   tensor(129.3969),
   tensor(-0.3997))],
 'cow': [(tensor(0.0003),
   tensor(157.8572),
   tensor(97.5015),
   tensor(161.3624),
   tensor(95.5378)),
  (tensor(4.0225e-05),
   tensor(161.5288),
   tensor(-0.7583),
   tensor(158.2054),
   tensor(1.8209))],
 'car': [(tensor(2.3265e-05),
   t

In [7]:
from src.utilities import get_ground_truth_objects_by_class

ground_truth_objects_by_class_1 = get_ground_truth_objects_by_class(y_batch[0])
ground_truth_objects_by_class_2 = get_ground_truth_objects_by_class(y_batch[1])

In [8]:
ground_truth_objects_by_class_1

{'cat': [(tensor(16.), tensor(43.5000), tensor(184.), tensor(224.5000))]}

In [9]:
ground_truth_objects_by_class_2

{'pottedplant': [(tensor(76.), tensor(28.), tensor(108.), tensor(70.))],
 'tvmonitor': [(tensor(119.), tensor(91.), tensor(153.), tensor(133.)),
  (tensor(152.), tensor(86.5000), tensor(186.), tensor(151.5000)),
  (tensor(185.5000), tensor(95.5000), tensor(214.5000), tensor(160.5000))],
 'chair': [(tensor(0.5000),
   tensor(100.5000),
   tensor(95.5000),
   tensor(223.5000))]}

In [10]:
tp_fp_by_class = {
    "aeroplane": [],
    "bicycle": [],
    "bird": [],
    "boat": [],
    "bottle": [],
    "bus": [],
    "car": [],
    "cat": [],
    "chair": [],
    "cow": [],
    "diningtable": [],
    "dog": [],
    "horse": [],
    "motorbike": [],
    "person": [],
    "pottedplant": [],
    "sheep": [],
    "sofa": [],
    "train": [],
    "tvmonitor": [],
}

In [11]:
class_object_totals = {
    "aeroplane": 0,
    "bicycle": 0,
    "bird": 0,
    "boat": 0,
    "bottle": 0,
    "bus": 0,
    "car": 0,
    "cat": 0,
    "chair": 0,
    "cow": 0,
    "diningtable": 0,
    "dog": 0,
    "horse": 0,
    "motorbike": 0,
    "person": 0,
    "pottedplant": 0,
    "sheep": 0,
    "sofa": 0,
    "train": 0,
    "tvmonitor": 0,
}

In [12]:
from src.evaluation import find_tp_and_fp

find_tp_and_fp(postprocessed_preds_1, ground_truth_objects_by_class_1, 
               tp_fp_by_class)

find_tp_and_fp(postprocessed_preds_2, ground_truth_objects_by_class_2, 
               tp_fp_by_class)

In [13]:
from src.evaluation import count_objects_in_each_class

count_objects_in_each_class(ground_truth_objects_by_class_1, class_object_totals)

count_objects_in_each_class(ground_truth_objects_by_class_2, class_object_totals)

In [14]:
tp_fp_by_class

{'aeroplane': [(tensor(3.8687e-05), False),
  (tensor(1.7779e-06), False),
  (tensor(4.4710e-05), False),
  (tensor(1.0915e-05), False)],
 'bicycle': [(tensor(0.0001), False),
  (tensor(0.0001), False),
  (tensor(0.0001), False),
  (tensor(0.0001), False)],
 'bird': [(tensor(5.3373e-05), False), (tensor(6.4678e-05), False)],
 'boat': [(tensor(0.0001), False), (tensor(0.0001), False)],
 'bottle': [(tensor(0.0001), False),
  (tensor(5.8679e-05), False),
  (tensor(0.0001), False),
  (tensor(5.7096e-05), False)],
 'bus': [],
 'car': [(tensor(1.4200e-05), False),
  (tensor(4.1835e-06), False),
  (tensor(2.3265e-05), False)],
 'cat': [],
 'chair': [(tensor(2.0832e-05), True)],
 'cow': [(tensor(0.0003), False),
  (tensor(4.3131e-05), False),
  (tensor(0.0003), False),
  (tensor(4.0225e-05), False)],
 'diningtable': [(tensor(0.0003), False),
  (tensor(0.0003), False),
  (tensor(4.0380e-05), False)],
 'dog': [(tensor(0.0002), False), (tensor(0.0002), False)],
 'horse': [(tensor(0.0001), False)]

In [18]:
class_object_totals

{'aeroplane': 0,
 'bicycle': 0,
 'bird': 0,
 'boat': 0,
 'bottle': 0,
 'bus': 0,
 'car': 0,
 'cat': 1,
 'chair': 2,
 'cow': 0,
 'diningtable': 0,
 'dog': 0,
 'horse': 0,
 'motorbike': 0,
 'person': 0,
 'pottedplant': 2,
 'sheep': 0,
 'sofa': 0,
 'train': 0,
 'tvmonitor': 6}

In [15]:
ground_truth_objects_by_class = get_ground_truth_objects_by_class(y_batch[1])
ground_truth_objects_by_class

{'pottedplant': [(tensor(76.), tensor(28.), tensor(108.), tensor(70.))],
 'tvmonitor': [(tensor(119.), tensor(91.), tensor(153.), tensor(133.)),
  (tensor(152.), tensor(86.5000), tensor(186.), tensor(151.5000)),
  (tensor(185.5000), tensor(95.5000), tensor(214.5000), tensor(160.5000))],
 'chair': [(tensor(0.5000),
   tensor(100.5000),
   tensor(95.5000),
   tensor(223.5000))]}

In [16]:
count_objects_in_each_class(ground_truth_objects_by_class, class_object_totals)

class_object_totals

{'aeroplane': 0,
 'bicycle': 0,
 'bird': 0,
 'boat': 0,
 'bottle': 0,
 'bus': 0,
 'car': 0,
 'cat': 1,
 'chair': 2,
 'cow': 0,
 'diningtable': 0,
 'dog': 0,
 'horse': 0,
 'motorbike': 0,
 'person': 0,
 'pottedplant': 2,
 'sheep': 0,
 'sofa': 0,
 'train': 0,
 'tvmonitor': 6}

In [17]:
precision_recall_lists = {
    "aeroplane": ([], []),
    "bicycle": ([], []),
    "bird": ([], []),
    "boat": ([], []),
    "bottle": ([], []),
    "bus": ([], []),
    "car": ([], []),
    "cat": ([], []),
    "chair": ([], []),
    "cow": ([], []),
    "diningtable": ([], []),
    "dog": ([], []),
    "horse": ([], []),
    "motorbike": ([], []),
    "person": ([], []),
    "pottedplant": ([], []),
    "sheep": ([], []),
    "sofa": ([], []),
    "train": ([], []),
    "tvmonitor": ([], []),
}

In [21]:
nums = [8, 4, 9, 3]

nums.sort(reverse=True)
nums

[9, 8, 4, 3]